# Polaris API traffic, v1 -- every operation, every reachable status. TRAFFIC ONLY.

**What this does.** Drives the same 286-cell grid as `polaris_log_coverage_v2.ipynb` -- the
same identities, fixture, doomed family, phases B-I and `Polaris-Request-Id`s -- plus phase J, a
table in a two-level namespace (2026-09-16) -- and **stops
there**. It records what was asked of Polaris and what
Polaris answered.

**What it does not do.** It verifies nothing. No OpenSearch, no VictoriaLogs, no `kubectl`, no
Lua unit test, no gates, no invariants, no findings. That is `local-k8s`'s job
(`PLAN-split-traffic-and-verification.md`); the last section prints the run id, each phase's
start/end time and the request ids that job needs.

**Import boundary.** Same rule as `src/make_traffic.py`: nothing here imports `os_report`,
`log_coverage` or `vlogs`. The traffic helpers come from `traffic_helpers` (which
`log_coverage` only re-exports -- the same functions).

`local` env only, and it **mutates**: a catalog, principals, roles, namespaces, a table and a
view are created and deleted, and 500s are provoked. ~300 calls.

**One window wait, then everything.** v2 slept to a report-window boundary before *every* phase.
This notebook waits **once**, just before phase B, and then drives phases B-I back to back, so
the whole drive starts at the top of a fresh window instead of wherever setup happened to end.
The last run took ~6 s for B-H, so at 30 s windows the drive fits in the window it starts in; the
summary says so if it did not. Each phase still records its start/end, and every call carries
its request id.

## 0 -- Setup

Polaris only. The run stops here if there is no root token.

In [1]:
import pathlib
import sys
import time
from datetime import datetime, timezone

_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from polaris_test_utils import *  # noqa: F401,F403
import polaris_test_utils as ptu
import api_status_matrix as mx
import traffic_helpers as th     # the traffic half of log_coverage, same functions
import make_traffic as mt        # window arithmetic only -- no logging concept

init_env("local")
require_not_prod("the api traffic drive creates and deletes catalogs, principals, roles, "
                 "namespaces, a table and a view, and provokes 500s")

RUN = str(int(time.time()))


def _iso(epoch):
    """UTC ISO-8601 with milliseconds -- how every phase records when it ran."""
    return datetime.fromtimestamp(epoch, tz=timezone.utc).isoformat(timespec="milliseconds")


STARTED = datetime.now(timezone.utc)

try:
    ROOT_TOK = root_token()
except Exception as exc:
    raise RuntimeError(f"SETUP FAILED -- Polaris at {ptu.POLARIS_URL}: "
                       f"{type(exc).__name__}: {exc}") from exc

print(f"  polaris  OK   {ptu.POLARIS_URL}  realm={ptu.REALM}")
print(f"  run={RUN}  started={STARTED.isoformat()}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.6.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.6.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
  polaris  OK   http://192.168.139.2:8181  realm=POLARIS
  run=1789955605  started=2026-09-21T01:53:25.036220+00:00


## 1 -- Identities

Four: admin (root), the runner, a denied principal with a role and no grants, and a garbage
bearer. The runner is elected in section 2, once the fixture exists to grant it rights on.

In [2]:
from polaris_rest import PolarisREST

adm_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)

RUNNER_NAME, RUNNER_ROLE = f"mx_{RUN}_runner", f"mx_{RUN}_runner_role"
DENIED_NAME, DENIED_ROLE = f"mx_{RUN}_denied", f"mx_{RUN}_denied_role"

TOKENS, IDENTITY_NOTES = {mx.ADMIN: ROOT_TOK}, []

# THE RUNNER IS ELECTED IN THE FIXTURE CELL, NOT HERE, and this is the whole
# lesson of run 1789007773. `provision_run_principal` grants the probe
# catalog's rights only when it is handed the fixture -- its own docstring says
# so: "catalog operations authorize against grant_records for the TARGET
# catalog and service_admin does not cover them". Provisioned here, before the
# fixture exists, the runner holds a role and no privileges, and 19 of the 2xx
# cells came back 403. The matrix then reported a Polaris that refuses every
# catalog operation, which is a statement about this cell.

_cid, _sec = th.provision_run_principal(adm_pc, DENIED_NAME, DENIED_ROLE, fixture=None,
                                        service_admin=False)
_r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{DENIED_ROLE}")
if _r.status_code == 200:
    TOKENS[mx.DENIED] = _r.json()["access_token"]
    IDENTITY_NOTES.append(f"denied  {DENIED_NAME} (a role, and no grants at all)")
else:
    TOKENS[mx.DENIED] = mx.GARBAGE_TOKEN
    IDENTITY_NOTES.append(f"denied  NO TOKEN ({_r.status_code}) -- every 403 cell will report "
                          "401 instead and be MISSED, correctly")

for n in IDENTITY_NOTES:
    print(f"  {n}")
print("  nobody  a syntactically valid bearer no realm accepts")

  denied  mx_1789955605_denied (a role, and no grants at all)
  nobody  a syntactically valid bearer no realm accepts


## 2 -- The fixture, the binding, and the doomed family

One catalog, namespace, table and view. `BINDING` turns a spec template into a URL. Every
destructive grid operation is pointed at the `doomed_` family so the run does not delete its own
fixture. `teardown_all` is registered here, with atexit, so an abort anywhere below still cleans up.

In [3]:
import api_surface as surf
from iceberg_rest import IcebergREST, build_create_table_payload, build_schema

adm_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)
run_adm_ic = adm_ic  # the doomed family is created as root; the grid deletes it as itself
SCHEMA = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

FX = surf.ProbeFixture(prefix=f"apimatrix{RUN}")
surf.setup_fixture(FX, adm_pc, adm_ic, ptu.BUCKET, ptu.MINIO_ENDPOINT, SCHEMA,
                   build_create_table_payload)
print(f"  fixture  {FX.cat} / {FX.ns} / {FX.tbl} / {FX.view}")

# NOW the runner can be elected: elect_drive_identity provisions it WITH the
# fixture (which is what grants the catalog rights) and then VERIFIES it with
# one management read and one catalog read, demoting to root loudly if it
# cannot do both. Run 1789007773 skipped both halves and spent 286 calls
# measuring a principal that could not read a table.
IDENTITY, run_pc, run_ic_elected, DRIVE_PRINCIPAL, IDENTITY_NOTES = th.elect_drive_identity(
    adm_pc, adm_ic, ptu.POLARIS_URL, ptu.REALM, RUNNER_NAME, RUNNER_ROLE, fixture=FX,
    root_token=ROOT_TOK)
TOKENS[mx.RUNNER] = run_pc.token or ROOT_TOK
for _n in IDENTITY_NOTES:
    print(f"  identity {_n}")
print(f"  runner   {DRIVE_PRINCIPAL} (elected: {IDENTITY})")
if DRIVE_PRINCIPAL == "root" or TOKENS[mx.RUNNER] == ROOT_TOK:
    print("  !! THE RUN IS DRIVING AS ROOT. That is a valid fallback and it VOIDS the")
    print("     per-principal margin: sum(principal.requests) == access_seen - parse_errors")
    print("     is then satisfied by a global counter with one row to sum, and proves nothing.")

BINDING = {
    "catalog": FX.cat, "namespace": FX.ns, "table": FX.tbl, "view": FX.view,
    "principalName": RUNNER_NAME, "principalRoleName": RUNNER_ROLE,
    "catalogRoleName": f"mx_{RUN}_crole", "plan-id": "no-such-plan-id",
    # names the grid CREATES rather than reads
    "new_catalog": f"mx{RUN}cat2", "new_namespace": f"mx_{RUN}_ns2",
    "new_table": f"mx_{RUN}_tbl2", "new_view": f"mx_{RUN}_vw2",
    "new_principal": f"mx_{RUN}_p2", "new_principal_role": f"mx_{RUN}_prole2",
    "new_catalog_role": f"mx_{RUN}_crole2",
    # THE RENAME FAMILY, separate from `new_*` since 2026-09-21. Phase B's
    # renameTable/renameView used to move `new_table`/`new_view` away, so
    # phase D's createTable and createView 409 cells had nothing left to
    # collide with -- run 1789950539 printed that as `{409: 403}` and
    # `{409: 200}`, two cells that could not pass, read as Polaris.
    "rename_table": f"mx_{RUN}_rn_tbl", "rename_view": f"mx_{RUN}_rn_vw",
    "renamed_table": f"mx_{RUN}_rn_tbl_renamed",
    "renamed_view": f"mx_{RUN}_rn_vw_renamed",
    # THE DOOMED FAMILY. Every destructive operation in the grid is pointed
    # here by api_status_matrix.REBIND, and nothing else in the run reads them.
    "doomed_catalog": f"mx{RUN}doomed", "doomed_namespace": f"mx_{RUN}_ns_doomed",
    "doomed_table": f"mx_{RUN}_tbl_doomed", "doomed_view": f"mx_{RUN}_vw_doomed",
    "doomed_principal": f"mx_{RUN}_p_doomed",
    "doomed_principal_role": f"mx_{RUN}_prole_doomed",
    "doomed_catalog_role": f"mx_{RUN}_crole_doomed",
    "entity_version": 1, "run": RUN,
    "base_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/",
    "allowed_location": f"s3a://{ptu.BUCKET}/",
    "s3_endpoint": ptu.MINIO_ENDPOINT, "s3_endpoint_internal": ptu.MINIO_ENDPOINT_INTERNAL,
    "metadata_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/never-registered/metadata.json",
    "client_id": ptu.ROOT_CLIENT, "client_secret": ptu.ROOT_SECRET,
}

# The catalog role the role-graph cells READ has to exist before they read it.
_r = adm_pc.create_catalog_role(FX.cat, BINDING["catalogRoleName"])
print(f"  catalog role {BINDING['catalogRoleName']}: {_r.status_code}")

# THE DOOMED FAMILY. The happy sweep drives deleteCatalog, dropTable, dropView,
# dropNamespace, deletePrincipal and the two revokes -- bound naively they take
# the FIXTURE, and the run destroys itself at about call 30 of 63 with every
# later cell 404ing. api_status_matrix.REBIND points them here instead, so the
# deletes are real deletes of real entities and nothing the run still needs.
SETUP = {}
SETUP["doomed catalog"] = adm_pc.create_catalog(
    BINDING["doomed_catalog"], ptu.BUCKET, ptu.MINIO_ENDPOINT,
    minio_endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
).status_code
SETUP["doomed namespace"] = run_adm_ic.create_namespace(
    FX.cat, BINDING["doomed_namespace"]).status_code
SETUP["doomed table"] = run_adm_ic.create_table(
    FX.cat, FX.ns, build_create_table_payload(BINDING["doomed_table"], SCHEMA)).status_code
SETUP["doomed view"] = run_adm_ic.create_view(FX.cat, FX.ns, {
    "name": BINDING["doomed_view"], "schema": SCHEMA,
    "view-version": {"version-id": 1, "timestamp-ms": 0, "schema-id": 0,
                     "summary": {"engine-name": "api-status-matrix"},
                     "default-namespace": [FX.ns],
                     "representations": [{"type": "sql", "sql": "SELECT 1",
                                          "dialect": "spark"}]},
    "properties": {"mx.run": RUN}}).status_code
SETUP["rename table"] = run_adm_ic.create_table(
    FX.cat, FX.ns, build_create_table_payload(BINDING["rename_table"], SCHEMA)).status_code
SETUP["rename view"] = run_adm_ic.create_view(FX.cat, FX.ns, surf.build_view_payload(
    BINDING["rename_view"], FX.ns, SCHEMA)).status_code
SETUP["doomed principal"] = adm_pc.create_principal(BINDING["doomed_principal"]).status_code
SETUP["doomed principal role"] = adm_pc.create_principal_role(
    BINDING["doomed_principal_role"]).status_code
SETUP["doomed catalog role"] = adm_pc.create_catalog_role(
    FX.cat, BINDING["doomed_catalog_role"]).status_code
# the two revokes need something assigned to revoke
SETUP["assign prole->principal"] = adm_pc.assign_principal_role_to_principal(
    BINDING["doomed_principal"], BINDING["doomed_principal_role"]).status_code
# ARGUMENT ORDER IS (catalog, principal_role, catalog_role) -- and this PUT can
# 404 immediately after create_catalog_role on PG-HA read-after-write lag, which
# is a known signature here, not a finding. Retry rather than record a false one.
SETUP["assign crole->prole"] = None
for _try in range(4):
    _ar = adm_pc.assign_catalog_role_to_principal_role(
        FX.cat, BINDING["doomed_principal_role"], BINDING["doomed_catalog_role"])
    SETUP["assign crole->prole"] = _ar.status_code
    if _ar.status_code != 404:
        break
    time.sleep(0.75)

for k, v in SETUP.items():
    print(f"  {k:26} {v}")
_bad = {k: v for k, v in SETUP.items() if not (200 <= int(v) < 300)}
if _bad:
    print(f"  !! setup did not fully succeed: {_bad}")
    print("     the delete cells for those entities will report 404 and be MISSED --")
    print("     correctly, and as a harness gap rather than a pipeline finding.")

# TEARDOWN THAT SURVIVES AN ABORT.
#
# Cleanup is section 11, eighteen cells below, and until now it was reachable
# only by running every cell between here and there. Any raise in between -- a
# gate, a transport error, an interrupted kernel -- left the catalog, the
# namespace, the table, the view, both principals and the whole doomed family
# on the cluster. That is where `apimatrix1789031469_cat` and `probe_ns` came
# from, and a leaked catalog is not tidiness: the next run's fixture collides
# with it and the collision is reported as a finding about Polaris.
#
# `teardown_all` is idempotent, never raises, and reads its clients out of
# globals(), so it is correct whether it runs from section 11, from atexit on
# kernel shutdown, or by hand in a fresh cell after a failure. Deletion order
# is children first: views and tables, then namespaces, then catalog roles,
# then catalogs. A 404 here is the expected answer for anything the grid
# already deleted as part of the test, and it is printed rather than hidden.
import atexit

_TORN = {"done": False}

# PHASE I IS TAGGED NOW, because section 11 says the cleanup DELETEs ARE the
# test -- five of the 63 operations -- and in run 1789950539 not one of them
# carried `nb-<run>-...`. Nineteen of the 186 kept access lines went out with
# a Quarkus-generated id, plus every 404ing attempt, so the verifier could not
# select phase I at all. A claim in a comment is not a tag.
#: Every request id phase I emits. Section 13 prints them with the rest, so
#: the verifier has the COMPLETE set of ids this run issued -- which is a
#: stronger check than a count, and the only one that can prove nothing was
#: dropped rather than that the totals happen to agree.
#:
#: ONE ID PER CALL, via `th.Tagger`. It used to be one id per GROUP, and run
#: 1789955605 measured what that costs: three ids carried two calls each -- the
#: black hole rung's cleanup and both deprovisions -- so the run could claim
#: "at least 345" against 354 seen and nothing could close the gap.
TEARDOWN_IDS = []


def teardown_all(quiet=False):
    """Delete everything this run created. Idempotent, best-effort, never raises."""
    if _TORN["done"]:
        if not quiet:
            print("  teardown already ran -- nothing to do")
        return {}
    _TORN["done"] = True
    g = globals()
    ic = g.get("run_ic") or g.get("adm_ic")
    pc = g.get("adm_pc")
    out = {}

    _clients = [c for c in (ic, pc) if c is not None]
    _TAG = th.Tagger(_clients, RUN, start=9000, prefix="teardown-")

    def _try(label, fn):
        try:
            out[label] = _TAG(label, fn).status_code
        except Exception as exc:
            out[label] = f"{type(exc).__name__}: {exc}"
        if not quiet:
            print(f"  {label:34} {out[label]}")

    if ic is not None:
        # THE RENAME DESTINATIONS, and leaving them out is what kept
        # `apimatrix1789365275_cat` on the cluster: renameTable/renameView
        # move `new_table`/`new_view` onto these names, nothing drops them,
        # so probe_ns answered 409 and the catalog delete 400. A teardown
        # that does not know every name the RUN can create is not a teardown.
        for _v in (FX.view, BINDING["doomed_view"], BINDING["new_view"],
                   BINDING["rename_view"], BINDING["renamed_view"]):
            _try(f"drop view {_v}", lambda v=_v: ic.drop_view(FX.cat, FX.ns, v))
        for _t in (FX.tbl, BINDING["doomed_table"], BINDING["new_table"],
                   BINDING["rename_table"], BINDING["renamed_table"]):
            _try(f"drop table {_t}", lambda t=_t: ic.drop_table(FX.cat, FX.ns, t))
        for _n in (BINDING["doomed_namespace"], BINDING["new_namespace"], FX.ns):
            _try(f"drop namespace {_n}", lambda n=_n: ic.drop_namespace(FX.cat, n))

    # THE SWEEP. Deleting a fixed list of names cannot be complete, because the
    # RUN creates entities it did not intend to. It now walks with
    # `surf.walk_namespaces`, DEEPEST FIRST, rather than listing the top level
    # once: `GET /namespaces` returns only the top level, children come from
    # `?parent=`, and the old sweep also joined a multi-level name with a dot
    # and tried to drop THAT -- so `probe_ns.nested` was looked for as a single
    # level called "probe_ns.nested" and was never going to be found.
    if ic is not None:
        try:
            _nss = surf.walk_namespaces(FX.cat, ic, tag=_TAG)
        except Exception as exc:
            _nss = []
            if not quiet:
                print(f"  {'sweep: walk namespaces':34} {type(exc).__name__}: {exc}")
        for _levels in _nss:
            _ns, _label = list(_levels), ".".join(_levels)
            for _kind, _lister, _dropper in (
                ("view", ic.list_views, ic.drop_view),
                ("table", ic.list_tables, ic.drop_table),
            ):
                try:
                    _r = _TAG(
                        f"sweep-list-{_kind}",
                        lambda l=_lister, n=_ns: l(FX.cat, n),
                    )
                    _ids = (_r.json().get("identifiers") or []) if _r.ok else []
                except Exception:
                    _ids = []
                for _id in _ids:
                    _nm = _id.get("name") if isinstance(_id, dict) else str(_id)
                    _try(f"sweep {_kind} {_label}/{_nm}",
                         lambda n=_ns, m=_nm, d=_dropper: d(FX.cat, n, m))
            _try(f"sweep namespace {_label}", lambda n=_ns: ic.drop_namespace(FX.cat, n))

    if pc is not None:
        # CATALOG ROLES ARE LISTED, NOT NAMED. The hardcoded list is what let
        # `{prefix}_shared` outlive the catalog until 9f3ae10, and a list is
        # only ever as complete as the last person to edit it. `catalog_admin`
        # is skipped by name: Polaris creates it with every catalog, refuses to
        # delete it on its own, and removes it with the catalog -- so attempting
        # it puts a permanent 400 in every clean teardown's output, and output
        # that always contains a failure is output nobody reads.
        try:
            _rr = _TAG("list-catalog-roles", lambda: pc.list_catalog_roles(FX.cat))
            _roles = ([r["name"] if isinstance(r, dict) else r
                       for r in (_rr.json().get("roles") or [])] if _rr.ok else [])
        except Exception as exc:
            _roles = []
            if not quiet:
                print(f"  {'list catalog roles':34} {type(exc).__name__}: {exc}")
        for _cr in _roles:
            if _cr == surf.BUILTIN_CATALOG_ROLE:
                out["builtin role skipped"] = _cr
                continue
            _try(f"delete catalog role {_cr}",
                 lambda c=_cr: pc.delete_catalog_role(FX.cat, c))

        for _cat in (FX.cat, BINDING["doomed_catalog"], BINDING["new_catalog"]):
            _try(f"delete catalog {_cat}", lambda c=_cat: pc.delete_catalog(c, purge=True))
            # SAY WHAT IS STILL IN IT. "Catalog X cannot be dropped, it is not
            # empty" names no entity, and this repo has chased that sentence
            # three times -- by hand on 2026-09-02, as `apimatrix1789366561_cat`,
            # and in run 1789955605, where the fixture catalog held on AFTER
            # `{prefix}_shared` was deleted 204 and every named namespace was
            # gone. AN EMPTY INVENTORY IS ITSELF THE FINDING: it means the
            # residue is invisible to the API that is supposed to manage it,
            # which is the shape a write that committed during a 500 leaves.
            if out.get(f"delete catalog {_cat}") not in (200, 204, 404):
                _left = _TAG(
                    f"remaining-in-{_cat}",
                    lambda c=_cat: surf.remaining_in_catalog(c, pc, ic),
                )
                out[f"remaining in {_cat}"] = _left
                if not quiet:
                    print(f"     STILL IN {_cat}: {_left}")
        for _p in (BINDING["doomed_principal"], BINDING["new_principal"]):
            _try(f"delete principal {_p}", lambda p=_p: pc.delete_principal(p))
        for _pr in (BINDING["doomed_principal_role"], BINDING["new_principal_role"]):
            _try(f"delete principal role {_pr}",
                 lambda p=_pr: pc.delete_principal_role(p))
        for _name, _role in ((RUNNER_NAME, RUNNER_ROLE), (DENIED_NAME, DENIED_ROLE)):
            try:
                th.deprovision_run_principal(pc, _name, _role, tag=_TAG)
                out[f"deprovision {_name}"] = "ok"
            except Exception as exc:
                out[f"deprovision {_name}"] = f"{type(exc).__name__}: {exc}"
            if not quiet:
                print(f"  {'deprovision ' + _name:34} {out['deprovision ' + _name]}")
    TEARDOWN_IDS.extend(_TAG.ids)
    return out


def _teardown_at_exit():
    if not _TORN["done"]:
        try:
            teardown_all(quiet=True)
        except Exception:
            pass


atexit.register(_teardown_at_exit)
print("  teardown registered: teardown_all() cleans up from anywhere, and atexit runs it on")
print("                       kernel shutdown if the run aborts before section 11")

  fixture  apimatrix1789955605_cat / probe_ns / probe_tbl / probe_view
  identity driving as mx_1789955605_runner (scope=PRINCIPAL_ROLE:ALL)
  runner   mx_1789955605_runner (elected: mx_1789955605_runner)
  catalog role mx_1789955605_crole: 201
  doomed catalog             201
  doomed namespace           200
  doomed table               200
  doomed view                200
  rename table               200
  rename view                200
  doomed principal           201
  doomed principal role      201
  doomed catalog role        201
  assign prole->principal    201
  assign crole->prole        201
  teardown registered: teardown_all() cleans up from anywhere, and atexit runs it on
                       kernel shutdown if the run aborts before section 11


## 3 -- The grid, printed BEFORE anything is driven

63 operations, 286 cells, from `log-coverage/spec/`. Every request is built once (dry run) before
any is issued, so a harness bug is found at call 0, not call 200.

In [4]:
import collections

# THE DENOMINATOR IS RECORDED, NOT ASSUMED.
#
# `spec/inventory.json` says which documents this grid was built from -- their
# sha256, the operation and cell counts they produce, and every fingerprint
# they superseded. `assert_denominator` hard-fails when the files on disk have
# moved since that record.
#
# This exists because of run 1789950539. It drove 63 operations / 286 cells at
# 00:29Z; `spec/` was re-fetched at a different tag at 01:00Z; the same call
# now answers 65 / 297. Nothing recorded the change, so that run's coverage
# figure is no longer reproducible from the tree -- and an unreproducible
# number still looks like a number. Re-record deliberately, in a commit that
# says why, never to turn a red run green.
_SPEC_DIR = pathlib.Path.cwd() / "spec"
SPEC = mx.assert_denominator(_SPEC_DIR)
print(f"  spec     {SPEC['tag']}   recorded {SPEC['recorded']}")
for _f in SPEC["files"]:
    print(f"           {_f['name']:32} {_f['sha256'][:16]}  {_f['bytes']}B")
if SPEC.get("history"):
    print(f"           {len(SPEC['history'])} superseded fingerprint(s) in inventory.json")

# The version the SERVER is, versus the version the config SAYS it is. The
# banner `init_env` prints is a string out of `src/config/<env>.yaml`, not a
# measurement -- run 1789950539 printed "Polaris 1.3.0" while driving a 1.6.0
# cluster that had been upgraded three days earlier, and every output of that
# run carries the wrong version label.
if str(ptu.POLARIS_VERSION) != str(SPEC.get("server_version")):
    print(f"  !! config says Polaris {ptu.POLARIS_VERSION}, inventory.json says the spec was "
          f"recorded for {SPEC.get('server_version')} -- one of them is stale, and the")
    print("     banner above is a CONFIG LABEL, never anything read from the server.")

OPS = mx.load_spec(_SPEC_DIR)
GRID = mx.cells(OPS)
BY_TARGET = collections.Counter(c.target for c in GRID)
BY_PHASE = collections.Counter(c.phase for c in GRID)

print(f"  {len(OPS)} operations from log-coverage/spec/  "
      f"({sum(1 for o in OPS if o.api == 'management')} management, "
      f"{sum(1 for o in OPS if o.api == 'catalog')} catalog)")
print(f"  {len(GRID)} cells   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_TARGET.items(), key=str)))
print("  phases   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_PHASE.items())))

print("\n  NOT REACHABLE on this build:")
for row in mx.ledger():
    mark = "settled" if row["state"] == "settled" else "PROBE  "
    print(f"    {str(row['status']):>4}  {mark}  {row['why']}")
print(f"  EXTRAS the spec never declares but this build returns: {mx.UNDECLARED_EXTRAS}")

# Every cell must build a request before ANY of them is issued. A missing payload
# or an unbindable path is a harness bug, and finding it at call 200 of 286 leaves
# a half-driven run on the cluster.
_dry = [mx.request_for(c, BINDING, TOKENS, "dry-run", ptu.REALM) for c in GRID]
print(f"\n  dry run  {len(_dry)} requests built, none raised")

ROWS, PHASES = [], {}


def run_phase(name, cells_, note=""):
    """Drive one phase immediately and record when it ran."""
    print(f"  phase {name}: {len(cells_)} cells {note}")
    t0 = time.time()
    rows = mx.drive(cells_, BINDING, TOKENS, ptu.POLARIS_URL, ptu.REALM, RUN,
                    seq_from=2000 + len(ROWS))
    t1 = time.time()
    PHASES[name] = {"start": _iso(t0), "end": _iso(t1),
                    "seconds": round(t1 - t0, 1), "cells": len(cells_)}
    ROWS.extend(rows)
    got = collections.Counter(r["verdict"] for r in rows)
    print(f"           {t1 - t0:.1f}s   {dict(got)}")
    return rows

  spec     apache-polaris-1.6.0   recorded 2026-09-21T01:25:46+00:00
           polaris-management-service.yml   9396ca3290f8e71f  58987B
           rest-catalog-open-api.yaml       ad52123061376b17  174023B
           1 superseded fingerprint(s) in inventory.json
  65 operations from log-coverage/spec/  (33 management, 32 catalog)
  297 cells   2=65  400=30  401=65  403=64  404=57  409=16
  phases   B=65  C=186  D=46

  NOT REACHABLE on this build:
     304  PROBE    needs If-None-Match support on loadTable
     406  PROBE    needs content negotiation to refuse a JSON-only endpoint
     419  PROBE    Iceberg's credential-refresh code; this build is expected to answer 401 instead
     429  settled  rateLimiter.type is no-op on this build -- the limiter cannot produce a 429
     502  settled  nothing proxies Polaris here; a port-forward is not a gateway
     503  settled  the only route is scaling Polaris, and HPA movement invalidates the run (local-k8s #8)
     504  settled  same -- no

## 3b -- Wait ONCE for a fresh window, then drive everything

The only wait in the notebook. Everything from here to phase I runs back to back -- **run the
remaining cells without pausing** (Restart & Run All, or Run All Below), because a pause between
cells is time inside the window.

`WINDOW_SECONDS` and `TICK_INTERVAL_S` are **parameters**, not read from the cluster; they must match
what `local-k8s` runs. A window is closed by the report tick, not by the clock (measured at
`window_end + 3.673 s`), so the drive starts `PHASE_LAG = TICK_INTERVAL_S + 1.5` s past the boundary
-- earlier than that and the first calls are booked to the previous window.

⚠ **30 / 5 are the TEMPORARY fast-run settings.** Change both when `local-k8s` reverts to 1800 / 30.

In [5]:
WINDOW_SECONDS = 30    # local-k8s report WINDOW_SECONDS   (fast-run; revert together)
TICK_INTERVAL_S = 5    # local-k8s polaris_report_tick Interval_Sec   (fast-run)

PHASE_LAG = TICK_INTERVAL_S + 1.5
assert PHASE_LAG < WINDOW_SECONDS / 2, (
    f"PHASE_LAG {PHASE_LAG}s leaves too little of a {WINDOW_SECONDS}s window to drive in")
# The tick's phase against the window grid is only fixed when the tick DIVIDES the
# window; otherwise the lag above stops meaning anything.
assert WINDOW_SECONDS % TICK_INTERVAL_S == 0, (
    f"Interval_Sec {TICK_INTERVAL_S} does not divide WINDOW_SECONDS {WINDOW_SECONDS}")

_wait = mt.seconds_to_boundary(WINDOW_SECONDS, lag=PHASE_LAG)
print(f"  waiting {_wait:.1f}s for a fresh {WINDOW_SECONDS}s window (+{PHASE_LAG}s past the boundary)")
time.sleep(_wait)

DRIVE_T0 = time.time()
DRIVE_WINDOW = mt.window_bounds(DRIVE_T0, WINDOW_SECONDS)[0]
print(f"  drive starts {_iso(DRIVE_T0)} in window {DRIVE_WINDOW}")

  waiting 8.9s for a fresh 30s window (+6.5s past the boundary)
  drive starts 2026-09-21T01:53:36.504+00:00 in window 2026-09-21T01:53:30Z


## 4 -- Phase B: the 2xx sweep

Every operation at its own declared success code (`createCatalog` 201, `dropNamespace` 204).
`verdict` here is Polaris-side only: did the status match the target.

In [6]:
PHASE_B = run_phase("B", [c for c in GRID if c.phase == "B" and c.target == 2])
_missed = [r for r in PHASE_B if r["verdict"] == mx.MISSED]
print(f"\n  {len(PHASE_B) - len(_missed)}/{len(PHASE_B)} covered")
for r in _missed[:20]:
    print(f"    MISS  {r['op_id']:34} got {r['status']}  {r['why']}")

  phase B: 65 cells 
           0.8s   {'covered': 54, 'missed': 11}

  54/65 covered
    MISS  registerTable                      got 403  expected one of (200,)
    MISS  registerView                       got 400  expected one of (200,)
    MISS  planTableScan                      got 404  expected one of (200,)
    MISS  cancelPlanning                     got 404  expected one of (204,)
    MISS  fetchPlanningResult                got 404  expected one of (200,)
    MISS  signRequest                        got 404  expected one of (200,)
    MISS  fetchScanTasks                     got 404  expected one of (200,)
    MISS  revokeGrantFromCatalogRole         got 400  expected one of (201,)
    MISS  revokeCatalogRoleFromPrincipalRole got 404  expected one of (204,)
    MISS  revokePrincipalRole                got 404  expected one of (204,)
    MISS  rotateCredentials                  got 403  expected one of (200,)


## 5 -- Phase C: 401 / 403 / 404

In [7]:
PHASE_C = run_phase("C", [c for c in GRID if c.phase == "C"])
_by = collections.Counter((r["target"], r["status"]) for r in PHASE_C)
print("\n  target -> actual:")
for (t, a), n in sorted(_by.items(), key=lambda kv: (str(kv[0][0]), str(kv[0][1]))):
    mark = "" if t == a else "   <-- MISS"
    print(f"    {t:>4} -> {str(a):>4}  x{n}{mark}")

  phase C: 186 cells 
           0.8s   {'covered': 164, 'missed': 22}

  target -> actual:
     401 ->  401  x60
     401 ->  404  x5   <-- MISS
     403 ->  200  x1   <-- MISS
     403 ->  403  x47
     403 ->  404  x16   <-- MISS
     404 ->  404  x57


## 6 -- Phase D: 409 / 400

409 from order on the creates and a stale `currentEntityVersion`; 400 from a body missing its
required fields, never from `{}`.

In [8]:
PHASE_D = run_phase("D", [c for c in GRID if c.phase == "D"])
for target in (409, 400):
    rows = [r for r in PHASE_D if r["target"] == target]
    hit = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    print(f"  {target}: {hit}/{len(rows)} covered")
    for r in rows:
        if r["verdict"] != mx.COVERED:
            print(f"    MISS  {r['op_id']:34} got {r['status']}")

  phase D: 46 cells 
           0.4s   {'covered': 27, 'missed': 19}
  409: 10/16 covered
    MISS  dropNamespace                      got 404
    MISS  updateTable                        got 200
    MISS  replaceView                        got 200
    MISS  renameTable                        got 404
    MISS  commitTransaction                  got 204
    MISS  renameView                         got 404
  400: 17/30 covered
    MISS  getToken                           got 500
    MISS  createNamespace                    got 500
    MISS  updateTable                        got 200
    MISS  planTableScan                      got 404
    MISS  signRequest                        got 404
    MISS  fetchScanTasks                     got 404
    MISS  replaceView                        got 200
    MISS  renameTable                        got 500
    MISS  renameView                         got 500
    MISS  createCatalogRole                  got 201
    MISS  createPrincipalRole            

## 7 -- Phase E: one table commit

Phase F's table is created first, so phase F has something to drop.

In [9]:
from iceberg_rest import IcebergREST as _IR

run_ic = _IR(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.RUNNER])

# Phase F's table, created here so phase F has something to drop.
# TAGGED. This create is a real request in the drive window and carried no
# request id until 2026-09-21, so it was one of the lines `access_seen` counted
# and `ISSUED` could not name.
_drop_tbl = f"mx_{RUN}_droponly"
E_SETUP_ID = mx.request_id(RUN, 3000, "gate2-the-droponly-create")
th.tag_clients([run_ic], E_SETUP_ID)
_r = run_ic.create_table(FX.cat, FX.ns, build_create_table_payload(_drop_tbl, SCHEMA))
th.tag_clients([run_ic], None)
print(f"  created {_drop_tbl}: {_r.status_code}  ({E_SETUP_ID})")

_t0 = time.time()
_rid = mx.request_id(RUN, 3001, "gate2-the-commit")
th.tag_clients([run_ic], _rid)
_resp = run_ic.commit_table(FX.cat, FX.ns, FX.tbl,
                            [{"action": "set-properties", "updates": {"mx.gate2": RUN}}])
th.tag_clients([run_ic], None)
_t1 = time.time()
GATE2 = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""),
    "t0": _t0, "t1": _t1, "at": _iso(_t0),
    "table": FX.tbl,
}
PHASES["E"] = {"start": _iso(_t0), "end": _iso(_t1), "seconds": round(_t1 - _t0, 1), "cells": 1}
print(f"  commit {GATE2['status']}, {GATE2['response_size']} bytes (as the client received it), "
      f"at {GATE2['at']}")

  created mx_1789955605_droponly: 200
  commit 200, 1941 bytes (as the client received it), at 2026-09-21T01:53:38.657+00:00


## 8 -- Phase F: one table delete

In [10]:
_t0 = time.time()
_rid = mx.request_id(RUN, 3002, "gate2-the-delete")
th.tag_clients([run_ic], _rid)
_resp = run_ic.drop_table(FX.cat, FX.ns, _drop_tbl)
th.tag_clients([run_ic], None)
_t1 = time.time()
GATE2_NEG = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""), "table": _drop_tbl,
    "t0": _t0, "t1": _t1, "at": _iso(_t0),
}
PHASES["F"] = {"start": _iso(_t0), "end": _iso(_t1), "seconds": round(_t1 - _t0, 1), "cells": 1}
print(f"  drop {GATE2_NEG['status']}, {GATE2_NEG['response_size']} bytes, at {GATE2_NEG['at']}")
if GATE2_NEG["response_size"] != 0:
    print("  !! the DELETE returned a non-empty body -- record it with the run.")

  drop 204, 0 bytes, at 2026-09-21T01:53:38.712+00:00


## 9 -- Phase G: three grants, and a denied read on the same role

`TABLE_READ_DATA`, `TABLE_WRITE_DATA`, `NAMESPACE_LIST` granted on one catalog role, then the
denied principal reads that role.

In [11]:
GRANTS = ["TABLE_READ_DATA", "TABLE_WRITE_DATA", "NAMESPACE_LIST"]
_crole = BINDING["catalogRoleName"]

_t0 = time.time()

granted = []
for i, priv in enumerate(GRANTS):
    _rid = mx.request_id(RUN, 3100 + i, f"gate4-grant-{priv}")
    th.tag_clients([adm_pc], _rid)
    r = adm_pc.grant_privilege(FX.cat, _crole, priv)
    th.tag_clients([adm_pc], None)
    granted.append({"privilege": priv, "status": r.status_code, "request_id": _rid})
    print(f"    grant {priv:22} {r.status_code}")

# The denied principal reads the role it has no grants on. No successful request
# to this role from this identity -- which is the whole point.
denied_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.DENIED])
_rid = mx.request_id(RUN, 3199, "gate4-denied-role-read")
th.tag_clients([denied_pc], _rid)
_dr = denied_pc.get_catalog_role(FX.cat, _crole)
th.tag_clients([denied_pc], None)
print(f"    denied read of the role: {_dr.status_code}")
_t1 = time.time()

GATE4 = {
    "role": _crole, "granted": granted, "t0": _t0, "t1": _t1,
    "expected_writes": sum(1 for g in granted if 200 <= (g["status"] or 0) < 300),
    "denied_status": _dr.status_code, "denied_request_id": _rid,
    "at": _iso(_t0),
}
PHASES["G"] = {"start": _iso(_t0), "end": _iso(_t1),
               "seconds": round(_t1 - _t0, 1), "cells": len(granted) + 1}
print(f"  successful grants on {_crole}: {GATE4['expected_writes']}  at {GATE4['at']}")

    grant TABLE_READ_DATA        201
    grant TABLE_WRITE_DATA       201
    grant NAMESPACE_LIST         201
    denied read of the role: 403
  successful grants on mx_1789955605_crole: 3  at 2026-09-21T01:53:38.755+00:00


## 10 -- Phase H: the 500 ladder

`th.provokers_500`. Every rung answered a 4xx on 2026-09-07, so NOT PROVOKED is the expected
result on this build.

In [12]:
# THE RUNG'S OWN FIXTURE CALLS ARE TAGGED, one id each. A rung's prepare makes
# two requests and its cleanup up to four, and until 2026-09-21 all of them
# went out under a single id for the whole group -- `drive_500` sees
# `tagged=True` and stops wrapping them.
LADDER_TAG = th.Tagger([adm_pc, run_ic], RUN, start=3210, prefix="probe-500-fixture-")
PROVOKERS = th.provokers_500(
    adm_pc, run_ic, RUN, bucket=ptu.BUCKET, endpoint=ptu.MINIO_ENDPOINT,
    endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
    table_payload=lambda name: build_create_table_payload(name, SCHEMA), repeat=3,
    tag=LADDER_TAG)
print(f"  ladder: {[p.name for p in PROVOKERS]}")

_t0 = time.time()
_r500 = th.drive_500([adm_pc, run_ic], RUN, 3200, PROVOKERS,
                     on_rung=lambda e: print(f"    {e['rung']:22} calls={e['calls']} "
                                             f"statuses={e['statuses']} provoked={e['provoked']}"))
_t1 = time.time()

FIVE_HUNDRED = {
    "winner": _r500["winner"], "rows": _r500["rows"], "ladder": _r500["ladder"],
    "driven_500": len(th.classify_500s(_r500["rows"])["deliberate"]),
    "at": _iso(_t0),
}
PHASES["H"] = {"start": _iso(_t0), "end": _iso(_t1),
               "seconds": round(_t1 - _t0, 1), "cells": len(_r500["rows"])}

if FIVE_HUNDRED["winner"]:
    print(f"  PROVOKED by `{FIVE_HUNDRED['winner']}`: {FIVE_HUNDRED['driven_500']} deliberate 500(s)")
else:
    print("  NOT PROVOKED -- no rung returned >= 500. Expected on this build: "
          "IcebergExceptionMapper\n     catches storage failures and maps them to 4xx.")

  ladder: ['black_hole_endpoint', 'unresolvable_host', 'nonexistent_bucket', 'stale_entity_version']
    black_hole_endpoint    calls=3 statuses=[500, 500, 500] provoked=3
  PROVOKED by `black_hole_endpoint`: 3 deliberate 500(s)


## 10b -- Phase J: a table in a two-level namespace

`th.drive_nested_namespace`: create `[<fixture ns>, "nested"]`, create `mx_<run>_deep` in it, commit
once, load once, drop the table, drop the namespace -- six calls as the runner, each with its own
request id. Driven back to back with B-H like every other phase.

**What the verifier looks for** (`local-k8s` TODO 1.5): Iceberg REST joins the levels with the unit
separator, which the path carries as `%1F`; the shipper keys the table row on that path, and rebuilds the
same key for commit time from the log line's dotted name `cat.ns.nested.table`. The run passes when the
report holds **one** row at the printed `resource_key` with both `requests` and `commit_count` -- a
second row means the two keys disagree. Nothing here checks that; it prints what to look for.

In [13]:
_t0 = time.time()
NESTED = th.drive_nested_namespace(run_ic, RUN, FX.cat, FX.ns, SCHEMA, build_create_table_payload)
_t1 = time.time()
PHASES["J"] = {"start": _iso(_t0), "end": _iso(_t1), "seconds": round(_t1 - _t0, 1),
               "cells": len(NESTED["rows"])}

for r in NESTED["rows"]:
    print(f"    {r['op_id']:16} {str(r['status']):>4} (want {r['target']})  {r['verdict']:8} "
          f"{r['request_id']}  {r.get('actual_path') or r.get('error', '')}")
print(f"  namespace {NESTED['namespace']}  table {NESTED['table']}")
print(f"  every table-level path carried %1F: {NESTED['path_encoded']}")
print(f"  report row to find: {NESTED['resource_key']}")
if not NESTED["path_encoded"]:
    print("  !! a path went out WITHOUT %1F -- the key above cannot match; record it with the run.")

    createNamespace   200 (want 200)  covered  nb-1789955605-3400-nested-createNamespace  /api/catalog/v1/apimatrix1789955605_cat/namespaces
    createTable       200 (want 200)  covered  nb-1789955605-3401-nested-createTable  /api/catalog/v1/apimatrix1789955605_cat/namespaces/probe_ns%1Fnested/tables
    updateTable       200 (want 200)  covered  nb-1789955605-3402-nested-updateTable  /api/catalog/v1/apimatrix1789955605_cat/namespaces/probe_ns%1Fnested/tables/mx_1789955605_deep
    loadTable         200 (want 200)  covered  nb-1789955605-3403-nested-loadTable  /api/catalog/v1/apimatrix1789955605_cat/namespaces/probe_ns%1Fnested/tables/mx_1789955605_deep
    dropTable         204 (want 204)  covered  nb-1789955605-3404-nested-dropTable  /api/catalog/v1/apimatrix1789955605_cat/namespaces/probe_ns%1Fnested/tables/mx_1789955605_deep
    dropNamespace     204 (want 204)  covered  nb-1789955605-3405-nested-dropNamespace  /api/catalog/v1/apimatrix1789955605_cat/namespaces/probe_ns%1Fnested
 

## 11 -- Phase I: cleanup

`dropTable`, `dropView`, `dropNamespace`, `deleteCatalog` and `deletePrincipal` are five of the 63
operations, so cleanup is traffic too. `teardown_all` is idempotent.

In [14]:
# The cleanup DELETEs are **part of the test**, not tidying: `dropTable`,
# `dropView`, `dropNamespace`, `deleteCatalog`, `deletePrincipal` are five of
# the 63 operations and rule 4 keeps every one of them. So they are run here,
# in the open, rather than left to the atexit hook.
#
# `teardown_all` was defined and registered in section 2, the moment the
# fixture existed, precisely so that an abort anywhere between there and here
# still cleans up. This cell is now the NORMAL path through it, not the only
# one. It is idempotent: if the hook already fired, it says so and does nothing.
CLEANUP = teardown_all()

# DID IT ACTUALLY CLEAN UP? Run 1789950539 printed
# `delete catalog apimatrix1789950539_cat 400` and nothing read it, so two
# catalogs stayed on the cluster and the next run's fixture had to collide
# with them before anyone noticed. A teardown that reports its own failures
# is the difference between a leak and a mystery.
_LEAKED = {k: v for k, v in CLEANUP.items()
           if k.startswith("delete catalog") and v not in (200, 204, 404)}
_LADDER_LEAKED = {e["rung"]: e["cleanup"] for e in FIVE_HUNDRED["ladder"]
                  if (e.get("cleanup") or {}).get("leaked")}
if _LEAKED or _LADDER_LEAKED:
    print("\n  !! CATALOGS LEFT ON THE CLUSTER -- record this with the run:")
    for _k, _v in _LEAKED.items():
        print(f"     {_k} -> {_v}")
        _cat = _k.removeprefix("delete catalog ").strip()
        _left = CLEANUP.get(f"remaining in {_cat}")
        if _left is not None:
            print(f"       still in it: {_left}")
            if _left.get("nothing_visible"):
                print("       NOTHING IS VISIBLE IN IT. Polaris says not-empty and the API")
                print("       that manages it can see nothing -- record this, it is a finding")
                print("       about the catalog, not about the teardown.")
    for _r, _c in _LADDER_LEAKED.items():
        print(f"     500-ladder rung {_r}: {_c}")
    print("     A catalog that outlives its run collides with the NEXT run's fixture,")
    print("     and that collision gets reported as a finding about Polaris.")
else:
    print("\n  teardown left nothing behind")

DRIVE_T1 = time.time()
DRIVE_END_WINDOW = mt.window_bounds(DRIVE_T1, WINDOW_SECONDS)[0]
print(f"\n  drive B-I: {DRIVE_T1 - DRIVE_T0:.1f}s, ended in window {DRIVE_END_WINDOW}"
      + ("" if DRIVE_END_WINDOW == DRIVE_WINDOW else
         f"  -- started in {DRIVE_WINDOW}, so it crossed a window boundary"))

  drop view probe_view               204
  drop view mx_1789955605_vw_doomed  404
  drop view mx_1789955605_vw2        204
  drop view mx_1789955605_rn_vw      404
  drop view mx_1789955605_rn_vw_renamed 204
  drop table probe_tbl               204
  drop table mx_1789955605_tbl_doomed 404
  drop table mx_1789955605_tbl2      204
  drop table mx_1789955605_rn_tbl    404
  drop table mx_1789955605_rn_tbl_renamed 204
  drop namespace mx_1789955605_ns_doomed 404
  drop namespace mx_1789955605_ns2   204
  drop namespace probe_ns            204
  sweep namespace probe_ns2          204
  delete catalog role mx_1789955605_crole 204
  delete catalog role mx_1789955605_crole_doomed 404
  delete catalog role mx_1789955605_crole2 204
  delete catalog role apimatrix1789955605_shared 204
  delete catalog apimatrix1789955605_cat 400
  delete catalog mx1789955605doomed  404
  delete catalog mx1789955605cat2    204
  delete principal mx_1789955605_p_doomed 404
  delete principal mx_1789955605_p2  204


## 12 -- The matrix, as Polaris answered it

One row per cell: `target` is what the cell asked for, `status` is what it got. A row where they
differ is MISSED with the actual code. Nothing here says anything about logs.

In [15]:
import pandas as pd

M = pd.DataFrame(ROWS)
RESULTS = {(r["op_id"], r["target"]): {"verdict": r["verdict"], "actual": r["status"],
                                       "why": r["why"]} for r in ROWS}
COVERAGE = mx.coverage_by_operation(OPS, RESULTS)

print("  by target:")
for target in sorted({r["target"] for r in ROWS}, key=str):
    rows = [r for r in ROWS if r["target"] == target]
    cov = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    err = sum(1 for r in rows if r["verdict"] == mx.ERROR)
    print(f"    {str(target):>4}  {cov:>3}/{len(rows):<3} covered"
          + (f"   {err} transport error(s)" if err else ""))

_full = [c for c in COVERAGE if not c["missed"] and not c["not_driven"]]
print(f"\n  operations with EVERY planned cell covered: {len(_full)}/{len(OPS)}")

_holes = [c for c in COVERAGE if c["missed"]]
print(f"  operations with a missed cell: {len(_holes)}")
for c in _holes[:25]:
    got = {t: RESULTS[(c['op_id'], t)]['actual'] for t in c['missed']}
    print(f"    {c['op_id']:34} missed {c['missed']}  got {got}")

pd.set_option("display.width", 200, "display.max_rows", 400, "display.max_colwidth", 44)
print("\n" + M[["op_id", "method", "target", "status", "verdict", "principal", "phase",
                "elapsed_ms"]].to_string(index=False))

  by target:
       2   54/65  covered
     400   17/30  covered
     401   60/65  covered
     403   47/64  covered
     404   57/57  covered
     409   10/16  covered

  operations with EVERY planned cell covered: 35/65
  operations with a missed cell: 30
    getConfig                          missed [403]  got {403: 200}
    getToken                           missed [400]  got {400: 500}
    createNamespace                    missed [400]  got {400: 500}
    dropNamespace                      missed [403, 409]  got {403: 404, 409: 404}
    registerTable                      missed [2]  got {2: 403}
    registerView                       missed [2]  got {2: 400}
    dropTable                          missed [403]  got {403: 404}
    updateTable                        missed [400, 409]  got {400: 200, 409: 200}
    planTableScan                      missed [2, 400, 401, 403]  got {2: 404, 400: 404, 401: 404, 403: 404}
    cancelPlanning                     missed [2, 401, 403]  got {2

## 13 -- The run, for the verifier

Everything `local-k8s` needs to find this run: the run id, when each phase ran, the request ids of the
single-purpose phases, and every >= 500 Polaris returned. No verdicts.

In [16]:
print(f"  run {RUN}  env={ptu.ENV}  polaris={ptu.POLARIS_URL}  realm={ptu.REALM}")
print(f"  started {STARTED.isoformat()}  finished {datetime.now(timezone.utc).isoformat()}")
print(f"  driven as {DRIVE_PRINCIPAL} (elected: {IDENTITY})")
print(f"  window_seconds={WINDOW_SECONDS}  tick={TICK_INTERVAL_S}s  drive started {PHASE_LAG}s "
      f"past a boundary")
print(f"  drive B-I  {_iso(DRIVE_T0)} .. {_iso(DRIVE_T1)}  windows {DRIVE_WINDOW} .. {DRIVE_END_WINDOW}"
      + ("" if DRIVE_END_WINDOW == DRIVE_WINDOW else "   CROSSED A BOUNDARY"))

print("\n  phases:")
for name, w in sorted(PHASES.items()):
    print(f"    {name}  {w['start']} .. {w['end']}  {w['seconds']:>6.1f}s  {w['cells']:>3} calls")

print("\n  single-purpose request ids:")
print(f"    E commit   {GATE2['request_id']}  status={GATE2['status']}  "
      f"bytes={GATE2['response_size']}  table={GATE2['table']}  at={GATE2['at']}")
print(f"    F delete   {GATE2_NEG['request_id']}  status={GATE2_NEG['status']}  "
      f"bytes={GATE2_NEG['response_size']}  table={GATE2_NEG['table']}  at={GATE2_NEG['at']}")
for g in GATE4["granted"]:
    print(f"    G grant    {g['request_id']}  status={g['status']}  {g['privilege']}  "
          f"role={GATE4['role']}")
print(f"    G denied   {GATE4['denied_request_id']}  status={GATE4['denied_status']}  "
      f"role={GATE4['role']}  at={GATE4['at']}")
for r in NESTED["rows"]:
    print(f"    J {r['op_id']:14} {r['request_id']}  status={r['status']}")
print(f"    J row      {NESTED['resource_key']}  (one row, requests AND commit_count)")

# ----------------------------------------------------------------------
# WHAT THIS RUN ISSUED, so the verifier can check that against what the
# window CONTAINS. Nothing in this notebook said so before, and nothing in
# `local-k8s` could therefore assert `access_seen == issued`: run 1789950539
# reported 337 access lines and the notebook offered no number to compare.
#
# The ids are the check, not the count. A total can agree by accident; a set
# of request ids cannot -- each one either appears in the window or does not,
# and the ones that do not are named.
# ----------------------------------------------------------------------
ISSUED = {
    "B/C/D grid": [r["request_id"] for r in ROWS],
    "E droponly create": [E_SETUP_ID],
    "E commit": [GATE2["request_id"]],
    "F delete": [GATE2_NEG["request_id"]],
    "G grants + denied read": [g["request_id"] for g in GATE4["granted"]]
    + [GATE4["denied_request_id"]],
    "H 500 ladder": [r["request_id"] for r in FIVE_HUNDRED["rows"]],
    "H ladder fixture": list(LADDER_TAG.ids),
    "J nested namespace": [r["request_id"] for r in NESTED["rows"]],
    "I teardown": list(TEARDOWN_IDS),
}
ALL_IDS = [rid for ids in ISSUED.values() for rid in ids]
print("\n  issued, by phase:")
for _name, _ids in ISSUED.items():
    print(f"    {_name:24} {len(_ids):>4}")
print(f"    {'TOTAL':24} {len(ALL_IDS):>4}   (all distinct: {len(set(ALL_IDS)) == len(ALL_IDS)})")

# ONE ID PER CALL, so this is an EQUALITY and not a floor. It was a floor
# until 2026-09-21 -- `at least N` -- because three helpers made two to four
# calls under one id: the 500 ladder's cleanup, and both deprovisions. Run
# 1789955605 showed 342 ids against 354 access lines and nothing could say
# where the other twelve came from.
print(f"\n  the window must hold EXACTLY {len(ALL_IDS)} access lines from this run,")
print("  one per id above, and every id must be found. The fixture setup is NOT")
print("  counted: it runs before section 3b's single wait, so it lands in an")
print("  earlier window than the drive.")

_by_principal = collections.Counter(r["principal"] for r in ROWS)
print("\n  grid calls by identity (the per-principal margin the report can be checked against):")
for _p, _n in sorted(_by_principal.items()):
    print(f"    {_p:10} {_n:>4}")
print("    NOTE: `nobody` is a garbage bearer, so Polaris logs it as `-`, and")
print("    `getToken` carries no bearer at all and is logged as `-` whoever drove it.")

ALL_500 = [r for r in ROWS if (r["status"] or 0) >= 500]
print(f"\n  >= 500 in the grid: {len(ALL_500)}   (ladder: "
      f"{FIVE_HUNDRED['winner'] or 'NOT PROVOKED'}, {FIVE_HUNDRED['driven_500']} deliberate)")
for r in ALL_500:
    print(f"    {r['op_id']:24} target {r['target']:>4} -> {r['status']}  "
          f"as {r['principal']}  phase {r['phase']}  {r['request_id']}")

  run 1789955605  env=local  polaris=http://192.168.139.2:8181  realm=POLARIS
  started 2026-09-21T01:53:25.036220+00:00  finished 2026-09-21T01:53:42.615076+00:00
  driven as mx_1789955605_runner (elected: mx_1789955605_runner)
  window_seconds=30  tick=5s  drive started 6.5s past a boundary
  drive B-I  2026-09-21T01:53:36.504+00:00 .. 2026-09-21T01:53:40.796+00:00  windows 2026-09-21T01:53:30Z .. 2026-09-21T01:53:30Z

  phases:
    B  2026-09-21T01:53:36.546+00:00 .. 2026-09-21T01:53:37.353+00:00     0.8s   65 calls
    C  2026-09-21T01:53:37.380+00:00 .. 2026-09-21T01:53:38.197+00:00     0.8s  186 calls
    D  2026-09-21T01:53:38.215+00:00 .. 2026-09-21T01:53:38.603+00:00     0.4s   46 calls
    E  2026-09-21T01:53:38.657+00:00 .. 2026-09-21T01:53:38.690+00:00     0.0s    1 calls
    F  2026-09-21T01:53:38.712+00:00 .. 2026-09-21T01:53:38.726+00:00     0.0s    1 calls
    G  2026-09-21T01:53:38.755+00:00 .. 2026-09-21T01:53:38.841+00:00     0.1s    4 calls
    H  2026-09-21T01:53:3

## 14 -- Teardown

Close the HTTP sessions and drop the frame so a re-run starts clean.

In [17]:
import gc

for _c in (adm_pc, adm_ic, run_ic, denied_pc):
    try:
        _c.session.close()
    except Exception:
        pass

del M
gc.collect()
print("  sessions closed, frame released")
print(f"\n  run {RUN} complete -- {len(ROWS)} grid calls, phases {sorted(PHASES)}")

  sessions closed, frame released

  run 1789955605 complete -- 297 grid calls, phases ['B', 'C', 'D', 'E', 'F', 'G', 'H', 'J']
